# System 3 — Fine-Tuning with LoRA
**Run this notebook on Google Colab with a T4 GPU.**
Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes -q

In [ ]:
# Upload finetune_data.jsonl from your data/ folder before running this cell
from google.colab import files
uploaded = files.upload()  # select data/finetune_data.jsonl

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType

# Load dataset
data = []
with open('finetune_data.jsonl') as f:
    for line in f:
        data.append(json.loads(line))

print(f'Loaded {len(data)} training examples')

def format_prompt(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

dataset = Dataset.from_list([{'text': format_prompt(d)} for d in data])
print(dataset[0]['text'][:200])

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
print('Model loaded.')

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

tokenized = dataset.map(tokenize, batched=True)

training_args = TrainingArguments(
    output_dir='./finetuned_model',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()
model.save_pretrained('./finetuned_model')
tokenizer.save_pretrained('./finetuned_model')
print('Fine-tuning complete!')

In [ ]:
# Test the fine-tuned model
def ask_finetuned(question):
    messages = [
        {'role': 'system', 'content': 'You are a programming tutor specializing in Data Structures, Algorithms, and Web Development.'},
        {'role': 'user', 'content': question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(ask_finetuned('What is binary search?'))

In [ ]:
# Download the fine-tuned model
import shutil
shutil.make_archive('finetuned_model', 'zip', './finetuned_model')
from google.colab import files
files.download('finetuned_model.zip')